# 07 - Real-World Validation

Every notebook up to this point measures performance on EMBER2018-derived CSVs, curated and pre-labelled. That is a different question from whether the model performs well on real files a user actually has on their machine. This notebook answers that question directly: it tests the deployed model against real `.exe`/`.dll` files and reports the result honestly.

**Not executed in this sandbox** (needs `pefile`, real local files, and the model file `06_evaluation.ipynb` produces). Run locally after `04`-`06`.


## Setup: load the deployed model

Loads the exact same file `app.py` loads, so this test matches exactly what a real user of the deployed app would experience.

In [1]:
# Standard library
import os
import sys

# Third-party
import joblib
import pandas as pd

pd.set_option("display.max_columns", None)
sys.path.append("../src")

from constants import ORDER_OF_FEATURES, LABEL_COLUMN
from explain import build_explainer, explain_prediction
from extract_features import extract_pe_features

MODEL_PATH = "../models/classical_pipeline.joblib"
DATA_PATH = "../data/dataset_pe_v2_full.csv"  # no separate train split file (see 02), use the full set for these reference averages
MALICIOUS_THRESHOLD = 0.5

In [2]:
artifact = joblib.load(MODEL_PATH)
pipeline = artifact["pipeline"]
feature_order = artifact["feature_order"]
print("model loaded, expects", len(feature_order), "features")

model loaded, expects 20 features


## Check 1: scan a folder assumed all-legitimate

Points at a real folder of known-legitimate files (e.g. `C:\Windows\System32`) and reports the false-positive rate, no ground-truth labels needed since every file there is legitimate by construction.

In [3]:
SCAN_FOLDER = r"C:\Windows\System32"  # edit to a real folder on your machine
SCAN_LIMIT = 200

In [4]:
scan_paths = []
for root, _, files in os.walk(SCAN_FOLDER):
    for fname in files:
        if fname.lower().endswith((".exe", ".dll")):
            scan_paths.append(os.path.join(root, fname))
        if len(scan_paths) >= SCAN_LIMIT:
            break
    if len(scan_paths) >= SCAN_LIMIT:
        break
print(f"found {len(scan_paths)} files to scan")

found 200 files to scan


In [5]:
scan_rows = []
for path in scan_paths:
    try:
        with open(path, "rb") as f:
            file_bytes = f.read()
        features = extract_pe_features(file_bytes, feature_order)
        proba = pipeline.predict_proba([features])[0][1]
        verdict = "malicious" if proba >= MALICIOUS_THRESHOLD else "benign"
        scan_rows.append({"path": path, "proba_malicious": proba, "verdict": verdict, "error": None})
    except Exception as e:
        scan_rows.append({"path": path, "proba_malicious": None, "verdict": None, "error": str(e)})

scan_results = pd.DataFrame(scan_rows)
print(f"scanned {len(scan_results)} files, {scan_results['error'].notna().sum()} could not be parsed")

C:\Users\darri\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\darri\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\darri\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\darri\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\darri\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\darri\anaconda3\Lib\site-p

scanned 200 files, 0 could not be parsed


C:\Users\darri\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\darri\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [6]:
scan_clean = scan_results[scan_results["verdict"].notna()]
fp_count = (scan_clean["verdict"] == "malicious").sum()
fp_rate = fp_count / len(scan_clean) if len(scan_clean) else float("nan")
print(f"False positives: {fp_count} / {len(scan_clean)} ({fp_rate:.1%})")

False positives: 3 / 200 (1.5%)


## Check 2: explain false positives with SHAP

Takes Check 1's results, picks the highest-confidence false positives, and shows a SHAP breakdown for each: which features pushed it toward malicious, compared against the typical benign/malicious training value for that feature. A value far outside both training averages usually signals a train/serve skew (the model never saw a real value like this during training), not that the value is inherently suspicious.

In [7]:
TOP_N = 5
flagged = scan_results[scan_results["verdict"] == "malicious"].sort_values("proba_malicious", ascending=False)
top_false_positives = flagged.head(TOP_N)
print(f"explaining top {len(top_false_positives)} false positives")

explaining top 3 false positives


In [8]:
train_df = pd.read_csv(DATA_PATH)
feature_stats = train_df.groupby(LABEL_COLUMN)[feature_order].mean()
explainer = build_explainer(pipeline)

for _, row in top_false_positives.iterrows():
    with open(row["path"], "rb") as f:
        file_bytes = f.read()
    features = extract_pe_features(file_bytes, feature_order)
    toward_malicious, _ = explain_prediction(explainer, pipeline, [features], feature_order)
    print(f"\n{row['path']} (p_malicious={row['proba_malicious']:.3f})")
    for feat, shap_val in toward_malicious[:3]:
        idx = feature_order.index(feat)
        val = features[idx]
        print(f"  {feat}: value={val}, benign_avg={feature_stats.loc[0, feat]:.2f}, "
              f"malicious_avg={feature_stats.loc[1, feat]:.2f}, shap={shap_val:.3f}")

C:\Users\darri\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\darri\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



C:\Windows\System32\ApproveChildRequest.exe (p_malicious=0.716)
  SectionMaxEntropy: value=7.291360481071174, benign_avg=6.28, malicious_avg=7.08, shap=1.615
  SectionMinRawsize: value=4096, benign_avg=42850.72, malicious_avg=21664.65, shap=1.042
  NumURLs: value=0, benign_avg=14.66, malicious_avg=8.77, shap=0.363

C:\Windows\System32\AutoCatHost.exe (p_malicious=0.566)
  SectionMaxEntropy: value=7.517766574872914, benign_avg=6.28, malicious_avg=7.08, shap=1.157
  AvgStringLength: value=6.5555, benign_avg=17.39, malicious_avg=43.23, shap=1.126
  SectionMinRawsize: value=4096, benign_avg=42850.72, malicious_avg=21664.65, shap=0.847

C:\Windows\System32\aitstatic.exe (p_malicious=0.528)
  SectionMaxEntropy: value=7.7356674588415055, benign_avg=6.28, malicious_avg=7.08, shap=1.888
  SectionMinRawsize: value=4096, benign_avg=42850.72, malicious_avg=21664.65, shap=0.520
  StringEntropy: value=5.0784, benign_avg=5.63, malicious_avg=5.99, shap=0.440


C:\Users\darri\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## Summary

Populated after a local run: the real false-positive rate found on `SCAN_FOLDER`, and the SHAP-based explanation for the highest-confidence false positives above.
